#Finding interactions in IDP chains.

The goal of this is to make a method that looks at all the chains in the csv file and get them for the proteins, to then compare the residues and see which ones are interacting within each protien.

I had to install the bio library as it was not included with Google Collab.

In [ ]:
!pip install Bio

In [ ]:
!pip install transformers

In [ ]:
!pip install datasets

Importing everything that is needed for the code to work effectively.

In [ ]:
# Importing everything that is needed
import re
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from io import StringIO
import Bio
import sklearn
from Bio.PDB import *
from transformers import AutoTokenizer
import torch
import datasets
from Bio.SCOP import *
from Bio.SeqUtils.ProtParam import ProteinAnalysis

I converted the xml file to a csv, as it was easier for me to work with. The file path goes to my google drive, so this might have to be changed. The columns dropped were empty and unneeded.

In [ ]:
# Reading the csv file
df = pd.read_csv("/content/drive/MyDrive/Coop/content/DIBS_complete.csv") # Make sure to change the path to wherever the file is actually located
# Unneeded columns
df = df.drop(columns=["/database", "Unnamed: 1", "Unnamed: 2"])

/tmp/ipykernel_18682/2016026119.py:2: DtypeWarning: Columns (3,15,16,17,18,26,27,29,30,31,32,33,35,37,38,41,42,43,44,46,47,50,51,58,59,61,62,63,64,66,67,70,71) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Coop/content/DIBS_complete.csv") # Make sure to change the path to wherever the file is actually located


The first row "/database" was empty and to make the code actually work, I had to get rid of it and adjust all the columns accordingly.

In [ ]:
# Fixes the gap in the dataframe in the first row caused by /database
new_header = df.iloc[0]
df = df[1:]
df.columns = new_header
pdb_col = "/entry/general/pdb_id"
chain_col = "/entry/evidence/chain_evidence/chain_id"

Because I had to store the csv file in google drive, I ran into some issues. Google drive uses a service called google sheets to store csv files, and tampered with the pdb file in a way that caused problems. It automatically converts strings like "4e05" to "400000". There were 2 other ids with this issue (3 in total). I tried hardcoding a solution but that resulted in errors in regards to chain collection. I decided to code a method to convert any instance of number conversion within the pdb ids, as hardcoding would be unfeasible for longer lists.

In [ ]:
pdb_pattern = re.compile(r"^[0-9][a-z0-9]{3}$", re.IGNORECASE)

def reconstruct_from_number(val):
    if pd.isna(val):
        return None


    if isinstance(val, str):
        s = val.strip()
        # If looks like scientific (e.g. "1E+91" or "1e91"), attempt to normalize:
        sci_match = re.match(r"^\s*([0-9]+)(?:\.[0-9]+)?[eE]\+?(-?\d+)\s*$", s)
        if sci_match:
            mant = sci_match.group(1)
            exp = int(sci_match.group(2))

            if len(mant) == 1 and 0 <= exp <= 99:
                candidate = f"{mant}e{exp:02d}"
                if pdb_pattern.match(candidate):
                    return candidate.lower()

            if len(mant) == 1 and 0 <= exp:
                candidate = f"{mant}e{exp}"
                if pdb_pattern.match(candidate):
                    return candidate.lower()
        # Checks if it already matches pdb pattern
        if pdb_pattern.match(s):
            return s.lower()
        # Checks if it looks like plain integer string
        try:
            # Attempt to parse as integers or gives up trying
            n = int(s.replace(",", ""))
        except Exception:
            return None
    else:
        # Converts to integer if it's an integer float
        if isinstance(val, float) and val.is_integer():
            n = int(val)
        elif isinstance(val, (int, np.integer)):
            n = int(val)
        else:
            try:
                n = int(val)
            except Exception:
                return None

    # Formats using Python's scientific formatting to get mantissa/exponent
    try:
        s_sci = "{:0.0e}".format(n)
    except Exception:
        return None

    # Removes plus sign and makes it lower-case
    s_sci = s_sci.replace("+", "").lower()
    m = re.match(r"^([0-9])e(-?\d+)$", s_sci)
    if m:
        mant = m.group(1)
        exp = int(m.group(2))
        if 0 <= exp <= 99:
            candidate = f"{mant}e{exp:02d}"
            if pdb_pattern.match(candidate):
                return candidate.lower()

            candidate2 = f"{mant}e{exp}"
            if pdb_pattern.match(candidate2):
                return candidate2.lower()
    # Fallback: if s_sci already length==4 and matches pattern, accept
    if len(s_sci) == 4 and pdb_pattern.match(s_sci):
        return s_sci.lower()

    return None

In [ ]:
df["pdb_fixed"] = df[pdb_col].apply(reconstruct_from_number)

def final_pdb_value(orig, fixed):
    if fixed is not None:
        return fixed
    if pd.isna(orig):
        return orig
    # Normalize original strings to lowercase and strip
    return str(orig).strip().lower()

df[pdb_col] = [final_pdb_value(o, f) for o, f in zip(df[pdb_col], df["pdb_fixed"])]

I made a list of all the ids of the protiens identified, the reason they had to be unique is because the way they are stored lead to a lot of repeats.

In [ ]:
pdb_ids = df["/entry/general/pdb_id"]
pdb_ids.head()
unique_pdb_ids = pdb_ids.unique()
unique_pdb_ids = unique_pdb_ids[1:]

In [ ]:
# Changing the old pdb_id column to the validated one
df_valid = df.dropna(subset=[chain_col]).copy()
df_valid[pdb_col]   = df_valid[pdb_col].astype(str).str.strip().str.lower()
df_valid[chain_col] = df_valid[chain_col].astype(str).str.strip()

df_valid = df_valid.drop_duplicates(subset=[pdb_col, chain_col], keep="first")

df_valid["_row_order"] = np.arange(len(df_valid))
df_valid = df_valid.sort_values("_row_order")

chain_vals = (
    df_valid
      .groupby(pdb_col, sort=False)[chain_col]
      .apply(list)
      .to_dict()
)
for pid in df[pdb_col].astype(str).str.strip().str.lower().unique():
    chain_vals.setdefault(pid, [])

df_valid.drop(columns=["_row_order"], inplace=True, errors="ignore")

In [ ]:
# How many chains there are per protein

chain_val_counts = {}

for i in chain_vals:
  if len(chain_vals[i]) not in chain_val_counts:
    chain_val_counts[len(chain_vals[i])] = 1
  else:
    chain_val_counts[len(chain_vals[i])] += 1

chain_val_counts

{2: 636, 3: 97, 5: 6, 4: 20}

In [ ]:
# Use the Bio library to reach the chains and residues to see if they interact
pdbl = PDBList()
parser = PDBParser(PERMISSIVE = True, QUIET = True)

# The Protein Class

I found that I needed a specific way to store proteins for a better way of analyzing them, so I created the protein class to store the itneractions and sequences that they all had.

In [ ]:
class Protein:
  def __init__(self, pdb_id):
    # Defines the structure and interactions, named by the id
    self.pdb_id = pdb_id
    self.structure = self.get_structure()
    self.chains = []
    self.interactions = []
    self.sequences = {}

# Returns the pdb_id
  def get_pdb_id(self):
    return self.pdb_id

# Gets the structure of the protein from the ".ent" file
  def get_structure(self):
    pdbl = PDBList()
    parser = PDBParser(PERMISSIVE = True, QUIET = True)
    structure = parser.get_structure(self.pdb_id, "/content/drive/MyDrive/Coop/content/pdb_ids/pdb"+self.pdb_id+".ent") # Change file path to where everything is stored
    return structure

# Uses the chains from the database to combine to store them all in a list
  def set_chains(self, list_of_chains):
    for i in list_of_chains:
      if i == self.pdb_id:
        self.chains.append(list_of_chains[i])
        break

  def get_chains(self):
    return self.chains

# Gets the list of residues with the option to remove the idp
  def get_residues(self, idp=True):
    if idp == True:
      return list(self.structure.get_residues())
    else:
      total_residues = []
      model = next(self.structure.get_models())

      chains_in_struct = [
              ch for chain_id in self.get_chains()[0]
              for ch in model if ch.id == chain_id
            ]
      for i in chains_in_struct[1:]:
        for j in i.get_residues():
          total_residues.append(j)
      return total_residues

# Get the chains from the bio python module, check if they interact, if they do, list where
  def set_interactions(self, cutoff=5.0):
        model = next(self.structure.get_models())  # get first model

        # Getting the chains in the structure
        chains_in_struct = [
              ch for chain_id in self.get_chains()[0]
              for ch in model if ch.id == chain_id
            ]

        if len(chains_in_struct) < 2:
            print("Not enough chains to compare.")
            return
        # Seperating between the idp and the other chain
        idp_chain = chains_in_struct[0]

        for other_chain in chains_in_struct[1:]:
            for res_a in idp_chain.get_residues():
                if res_a.id[0] != " ":
                    continue
                for res_b in other_chain.get_residues():
                    if res_b.id[0] != " ":
                        continue

                    interacting = False
                    for atom_a in res_a:
                        for atom_b in res_b:
                            distance = atom_a - atom_b  # Biopython atom distance
                            if distance <= cutoff:
                                self.interactions.append((res_a.get_resname(), idp_chain.id, res_a.id[1], res_b.get_resname(), other_chain.id, res_b.id[1]))
                                interacting = True
                                break
                        if interacting:
                            break

  # Gets the interactions, Ex: (IDP residue, IDP chain ID, idp residue ID, Residue, other chain ID, other chain ID)
  def get_interactions(self):
    return self.interactions

  # Sets all the sequences of a protein into a dictionary and seperates them by chain
  def set_sequences(self):
    parser = PDBParser(QUIET=True)
    ppb = PPBuilder()
    model = next(self.structure.get_models())
    for chain in model:
            chain_id = chain.get_id()

            polypeptides = ppb.build_peptides(chain)

            if polypeptides:
                # Concatenate all polypeptide sequences in the chain
                chain_sequence = "".join([str(pp.get_sequence()) for pp in polypeptides])
                self.sequences[chain_id] = chain_sequence

  # Gets the sequences of a protein, seperated by chain
  def get_sequences(self):
    return self.sequences



I compiled all of the protiens into a list that operates like a pseudo database. The order comes from how they are stored in the csv file.

In [ ]:
protien_list = []
for i in unique_pdb_ids:
  protien_list.append(Protein(i))
  protien_list[-1].set_chains(chain_vals)
  protien_list[-1].set_interactions()
  protien_list[-1].set_sequences()

This was for viewing purposes, to see all the interactions occured between the idp within each protein.

In [ ]:
print(protien_list[0].get_interactions())
print(protein_list[0].get_chains())
print(protein_list[0].get_sequences())
print(protein_list[0].get_residues())

# Interaction Matrix

The interaction matrix was created to have a visual representation of all the residues that do interact with the idp within each protein.

In [ ]:
# Getting the residue names from all the interactions in the idps

idp_res_name_counts = {}

for i in protien_list:
  for j in i.get_interactions():
    if j[0] not in idp_res_name_counts:
      idp_res_name_counts[j[0]] = 1
      continue
    idp_res_name_counts[j[0]] += 1

idp_res_name_counts

In [ ]:
binding_res_name_counts = {}

for i in protien_list:
  for j in i.get_interactions():
    if j[3] not in binding_res_name_counts:
      binding_res_name_counts[j[3]] = 1
      continue
    binding_res_name_counts[j[3]] += 1

binding_res_name_counts

Using the list of interactions, I created a seperate matrix class that shows the interactions that the residues have.

In [ ]:
idp_res_names = list(idp_res_name_counts.keys())
print(idp_res_names)
binding_res_names = list(binding_res_name_counts.keys())
print(binding_res_names)

In [ ]:
# Making a matrix class to keep track of The interactions

class Matrix:
  def __init__(self, pdb_id, row=idp_res_names, col=binding_res_names, matrix_size=20):

    self.pdb_id = pdb_id
    self.row_head = row
    self.col_head = col
    self.contents = np.zeros((matrix_size, matrix_size))
    self.matrix_size = matrix_size

  # The modified string definition is used for printing purposes
  def __str__(self):
    str_matrix = self.pdb_id + "\n"
    h = 0
    while h < self.matrix_size:
      str_matrix += f"{str(self.row_head[h])} "
      h += 1
    i = 0
    while i < self.matrix_size:
      str_matrix += f"\n{str(self.col_head[i])} {self.contents[i]}"
      i += 1
    return str_matrix

  # Sets the row head of the matrix
  def set_row_head(self, content):
    self.row_head = content
  # Sets the column of the matrix
  def set_col_head(self, content):
    self.col_head = content

  # Adds a interaction to the matrix via row and column
  def add_matrix_interaction(self, row_name, column_name):
      for i in self.row_head:
        if i == row_name:
          for j in self.col_head:
            if j == column_name:
              self.contents[self.row_head.index(i)][self.col_head.index(j)] += 1
              return


In [ ]:
# Test to see if the matrix actually worked
matrix = Matrix("test")
matrix.add_matrix_interaction("ALA", "MET")
print(matrix)

In [ ]:
matrix_list = []
for i in protien_list:
  matrix = Matrix(i.get_pdb_id())
  for j in i.get_interactions():
    matrix.add_matrix_interaction(j[0], j[3])
  matrix_list.append(matrix)

In [ ]:
# Example matrix of an actual protein

print(matrix_list[0])

In [ ]:
print(binding_res_names)

# Statistics of Protien Frequency

Using the residues that I had found that binded to the idp, I constructed bargraphs containing the frequencies of the residues interact occur.

In [ ]:
# Getting the total amount of binding resnames that occur in the interactions
print(binding_res_name_counts)
binding_res_name_sum = 0
for i in binding_res_name_counts:
  binding_res_name_sum += binding_res_name_counts[i]
print(binding_res_name_sum)

binding_res_names_inter_freqs = {}
for i in binding_res_name_counts:
  binding_res_names_inter_freqs[i] = binding_res_name_counts[i] / binding_res_name_sum
print(binding_res_names_inter_freqs)

In [ ]:
# Making a figure for the frequncies of interactions that occur in each protein.
fig = plt.figure(figsize = (10, 5))
fig.set_facecolor("white")

plt.title("Interaction Frequencies of amino acids")
plt.xlabel("Amino Acid")
plt.ylabel("Interaction Frequency")

plt.bar(binding_res_names_inter_freqs.keys(), binding_res_names_inter_freqs.values())

In [ ]:
# Getting the total amount of binding resnames that occur in the proteins (including those that do not bind)

total_binding_res_name_counts = {}

for i in protien_list:
  for j in i.get_residues(False):
    if j.get_resname() not in total_binding_res_name_counts:
      total_binding_res_name_counts[j.get_resname()] = 1
      continue
    total_binding_res_name_counts[j.get_resname()] += 1



The total binding residues had contained residues that were not in the interacting set. Regardless, it was important to keep track of them, otherwise the data would be flawed.

In [ ]:
print(total_binding_res_name_counts)

In [ ]:
total_binding_res_sum = 0
for i in total_binding_res_name_counts:
  if i in binding_res_names_inter_freqs:
    total_binding_res_sum += total_binding_res_name_counts[i]

print(total_binding_res_sum)

In [ ]:
total_binding_res_freqs = {}

for i in total_binding_res_name_counts:
  if i in binding_res_names_inter_freqs:
    total_binding_res_freqs[i] = total_binding_res_name_counts[i] / total_binding_res_sum

print(total_binding_res_freqs)

The following graph shows the frequency of amino acids that appear in the proteins in the database.

In [ ]:
fig = plt.figure(figsize = (10, 5))
fig.set_facecolor("white")

plt.title("Total Frequency of amino acids")
plt.xlabel("Amino Acid")
plt.ylabel("Total Frequency")

plt.bar(total_binding_res_freqs.keys(), total_binding_res_freqs.values())

In [ ]:
# Getting the logarithmic frequencies

ln_freqs = {}

for i in total_binding_res_freqs:
    ln_freqs[i] = np.log(binding_res_names_inter_freqs[i] / total_binding_res_freqs[i])
print(ln_freqs)

The ln Frequency takes the logarithmic (log10) of the frequency of amino acids. This shows a pattern of whether or not there were more or less interacting residues than the total amount of them.

In [ ]:
# Making the bar graph

fig = plt.figure(figsize = (10, 5))
fig.set_facecolor("white")

plt.title("Ln Frequency of amino acids")
plt.xlabel("Amino Acid")
plt.ylabel("Ln Frequency")

plt.bar(ln_freqs.keys(), ln_freqs.values())

In [ ]:
# Normalized frequencies via min max normalization

min = 0
max = 0

for i in ln_freqs:
  if ln_freqs[i] < min:
    min = ln_freqs[i]
  if ln_freqs[i] > max:
    max = ln_freqs[i]

print(min, max)

In [ ]:
sum = 0
for i in ln_freqs:
  sum += ln_freqs[i]

print(sum)

In [ ]:
norm_freqs = {}

for i in ln_freqs:
  norm_freqs[i] = ((ln_freqs[i] - min / (max - min))) / sum

print(norm_freqs)

The following graph shows the normalized frequencies created via min-max normalization of the ln frequences. This provides insight into the ln frequencies without bias.

In [ ]:
fig = plt.figure(figsize = (10, 5))
fig.set_facecolor("white")

plt.title("Normalized Frequencies")
plt.xlabel("Amino Acid")
plt.ylabel("Normalized Frequency")

plt.bar(norm_freqs.keys(), norm_freqs.values())

In [ ]:
# Figuring out which amino acids have a greater total compared to interaction ratio

for i in total_binding_res_freqs:
  if i in binding_res_names_inter_freqs:
    if binding_res_names_inter_freqs[i] > total_binding_res_freqs[i]:
      print(i+" has a greater frequency of binding than total")
    elif binding_res_names_inter_freqs[i] < total_binding_res_freqs[i]:
      print(i+" has a greater frequency of total than binding")
    else:
      print(i+" has equal frequncies")
  else:
    continue

This was made to verify that both frequencies add up to (around) 1, validating the data.

In [ ]:
inter_sum = 0

for i in binding_res_names_inter_freqs:
  inter_sum += binding_res_names_inter_freqs[i]
print(inter_sum)

In [ ]:
total_sum = 0

for i in total_binding_res_freqs:
  total_sum += total_binding_res_freqs[i]
print(total_sum)